In [0]:
# Capstone Project: End-to-End MLOps Pipeline
# Phase 1: Data Ingestion and Exploration

""" 
This notebook loads a binary classification dataset into Databricks, stores the dataset in DBFS, performs exploratory data analysis using PySpark DataFrames and the Pandas API on Spark, and documents the schema, feature distributions, and preprocessing plan.

Dataset: Breast Cancer Wisconsin classification dataset  
Target column: `target`  
Problem type: Binary classification 
"""

In [0]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import re

# Load classification dataset
data = load_breast_cancer(as_frame=True)
pdf = data.frame

display(pdf.head())

# Clean pandas column names before converting to Spark
def clean_column_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

pdf.columns = [clean_column_name(c) for c in pdf.columns]

display(pdf.head())

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(pdf)

# Check current catalog
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
print(f"Current catalog: {current_catalog}")

# Use current catalog and default schema
catalog = current_catalog
schema = "default"
table_name = "breast_cancer"

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

full_table_name = f"{catalog}.{schema}.{table_name}"

# Write cleaned Spark DataFrame to Delta table
(
    spark_df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Dataset saved to table: {full_table_name}")
print(f"Rows: {pdf.shape[0]}")
print(f"Columns: {pdf.shape[1]}")

# Confirm table loads correctly
saved_df = spark.table(full_table_name)
display(saved_df)
saved_df.printSchema()

In [0]:
# Read from Unity Catalog table (created in cell 2)
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
spark_df = spark.table(f"{current_catalog}.default.breast_cancer")

display(spark_df)
spark_df.printSchema()

In [0]:
from pyspark.sql.functions import col, count, isnan, when

# Summary statistics
display(spark_df.describe())

# Missing values by column
missing_df = spark_df.select([
    count(
        when(
            col(c).isNull() | isnan(col(c)),
            c
        )
    ).alias(c)
    for c in spark_df.columns
])

display(missing_df)

# Target distribution
display(spark_df.groupBy("target").count())

In [0]:
import pyspark.pandas as ps

psdf = spark_df.pandas_api()

display(psdf.head())
display(psdf.describe())

In [0]:
import matplotlib.pyplot as plt

sample_pdf = spark_df.toPandas()

selected_columns = [
    "mean_radius",
    "mean_texture", 
    "mean_perimeter",
    "mean_area",
    "target"
]

sample_pdf[selected_columns].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [0]:
# Data Schema

"""
The dataset contains numeric measurements computed from digitized images of breast masses. Each row represents one observation.

The target column is `target`.

- `0`: malignant
- `1`: benign
"""

# Initial Data Quality Findings
"""
- The dataset was loaded from scikit-learn and saved to DBFS.
- The schema was inferred by Spark.
- Missing values were checked across all columns.
- Target class distribution was reviewed to understand class balance.
"""
# DBTITLE 1,Data Schema
# The dataset contains numeric measurements computed from digitized images of breast masses. Each row represents one observation.

# The target column is `target`.

# - `0`: malignant

# Preprocessing Plan
"""
The model training notebook will:

1. Convert the Spark DataFrame to a Pandas DataFrame.
2. Separate features from the target column.
3. Split the dataset into training and test sets.
4. Scale numeric features with `StandardScaler`.
5. Train several classifier configurations.
6. Track all runs with MLflow.
7. Register the best model in Unity Catalog.
8. Deploy the champion model to Databricks Model Serving.
"""